# Lección 4.8 — Elegir la métrica según el costo del error en el negocio

> **Módulo 4 · Evaluación: ¿mi modelo es bueno?** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lección 4.2 (matriz de confusión, precisión, *recall*), Lección 4.3 (umbrales), Lección 4.6 (validación cruzada), Lección 4.7 (líneas base)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Traducir un problema de negocio a una **matriz de costos** (o de beneficios).
- Calcular el **costo esperado** de un modelo y compararlo con líneas base.
- Deducir el **umbral óptimo** a partir de los costos, sin fórmulas complicadas.
- Usar **F-beta** cuando no conoces los costos en dinero, pero sí cuál error es peor.
- Ajustar el umbral **sin hacer trampa** con `TunedThresholdClassifierCV`.
- Elegir la métrica adecuada para un problema de regresión o de clasificación.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split, cross_val_predict,
                                     TunedThresholdClassifierCV, FixedThresholdClassifier)
from sklearn.metrics import (confusion_matrix, make_scorer, accuracy_score,
                             recall_score, precision_score, fbeta_score)

plt.rcParams["figure.figsize"] = (9, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: la alarma de humo

Una alarma de humo puede equivocarse de dos maneras:
- **Falsa alarma:** suena porque se te quemó el pan tostado. Molesto: abres la ventana y ya.
- **Alarma silenciosa:** hay un incendio de verdad y **no suena**. Catastrófico.

Por eso los fabricantes la hacen **muy sensible**: prefieren 50 falsas alarmas a una sola alarma silenciosa. Nadie evalúa una alarma de humo por su "exactitud".

Todos los modelos de clasificación tienen estos dos tipos de error (Lección 4.2): **falsos positivos** (FP) y **falsos negativos** (FN). La pregunta de esta lección es: **¿cuánto cuesta cada uno en tu negocio?** La respuesta decide qué métrica usar y dónde poner el umbral (Lección 4.3).

### ✋ Ejercicio 1 — ¿Qué error duele más?
Para cada caso, di qué es un FP y qué es un FN, y cuál es **más caro**:

1. Un filtro de spam (positivo = "es spam").
2. Una prueba de detección temprana de cáncer (positivo = "posible cáncer, haz más estudios").
3. Un sistema que decide a qué clientes enviar un cupón de $2 (positivo = "le enviamos el cupón").

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **FP:** un correo importante termina en spam. **FN:** te llega un spam. El **FP es peor**: perder el correo de una oferta de trabajo duele más que borrar un spam. Por eso los filtros de spam son "conservadores".
2. **FP:** una persona sana se hace estudios extra (susto, dinero). **FN:** un cáncer pasa sin detectarse. El **FN es muchísimo peor**: aquí se busca un *recall* altísimo.
3. **FP:** mandas un cupón a quien no iba a comprar: pierdes $2. **FN:** no se lo mandas a alguien que sí habría comprado: pierdes la ganancia de la venta. Depende de cuánto ganes por venta, pero normalmente el **FN es más caro**.

No hay una métrica "correcta" en abstracto: **depende de lo que cuesta cada error**.
</details>

---
## 2. 📖 La matriz de costos

Pon precio a cada casilla de la matriz de confusión. Caso: una planta industrial tiene **4,000 bombas** con sensores y quiere predecir cuáles **fallarán la próxima semana** para revisarlas antes.

|  | **Predice: no revisar** | **Predice: revisar** |
|---|---|---|
| **Realmente no iba a fallar** | ✅ VN: $0 | ❌ FP: revisión innecesaria → **$1,000** |
| **Realmente iba a fallar** | ❌ FN: la bomba se rompe, la línea se para → **$20,000** | ✅ VP: falla evitada → $0 * |

\* Simplificación: contamos solo el **costo de equivocarse**. La revisión de una bomba que sí iba a fallar cuesta $1,000, pero evita una avería de $20,000; la tratamos como "lo correcto, sin pérdida". En el Ejercicio final verás qué pasa si no simplificas.

Con esta tabla, el **costo promedio por bomba** de un modelo es:

$$\text{costo} = \frac{\text{FP} \times 1{,}000 + \text{FN} \times 20{,}000}{\text{número de bombas}}$$

Un FN cuesta **20 veces** más que un FP.

In [ ]:
X, y = make_classification(n_samples=4000, n_features=6, n_informative=4, n_redundant=1,
                           weights=[0.92], class_sep=0.8, flip_y=0.02, random_state=7)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
print(f"Bombas que fallan: {y.mean():.1%}")

C_FP, C_FN = 1_000, 20_000      # costo de una revisión innecesaria y de una falla no detectada

def costo(y_real, y_pred, c_fp=C_FP, c_fn=C_FN):
    """Costo promedio por bomba (en $)."""
    vn, fp, fn, vp = confusion_matrix(y_real, y_pred, labels=[0, 1]).ravel()
    return (fp * c_fp + fn * c_fn) / len(y_real)

### 🔮 Ejercicio 2 — ¿Quién gasta menos?
Comparamos tres estrategias con validación cruzada sobre el entrenamiento (Lección 4.6):
- **Nunca revisar** (línea base, Lección 4.7).
- **Revisar todas** las bombas (otra línea base).
- Una **regresión logística** con el umbral por defecto (0.5).

Antes de ejecutar: ¿cuál será la más barata? ¿Y cuál tendrá la mayor **exactitud**?

✍️ *Mis predicciones:*

In [ ]:
modelo = make_pipeline(StandardScaler(), LogisticRegression())
prob = cross_val_predict(modelo, X_tr, y_tr, cv=5, method="predict_proba")[:, 1]   # prob. de falla

estrategias = {
    "Nunca revisar":          np.zeros_like(y_tr),
    "Revisar todas":          np.ones_like(y_tr),
    "Logística (umbral 0.5)": (prob >= 0.5).astype(int),
}
for nombre, pred in estrategias.items():
    print(f"{nombre:<24} exactitud = {accuracy_score(y_tr, pred):5.1%}   "
          f"recall fallas = {recall_score(y_tr, pred):4.0%}   costo por bomba = ${costo(y_tr, pred):,.0f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Nunca revisar:** exactitud **91.2 %** (¡la segunda más alta!), pero cuesta **$1,767** por bomba: todas las fallas se pagan completas.
- **Revisar todas:** exactitud **8.8 %**, horrible… y sin embargo cuesta **$912**, mucho menos.
- **Logística con 0.5:** exactitud **~93 %**, la mejor, pero cuesta **$1,315**: solo detecta el 26 % de las fallas.

El modelo con mejor exactitud es **más caro** que la estrategia con peor exactitud. Con costos tan desiguales, la exactitud **mide lo que no importa**. El problema no es el modelo: es el **umbral** de 0.5, que trata igual a los dos errores.
</details>

---
## 3. 📖 ¿Dónde pongo el umbral? Lo deduces tú

El modelo dice: *"esta bomba tiene una probabilidad **p** de fallar"*. Tienes dos opciones:

| Si decides… | Te equivocas cuando… | Costo esperado |
|---|---|---|
| **Revisar** | no iba a fallar (prob. 1 − p) | (1 − p) × 1,000 |
| **No revisar** | sí iba a fallar (prob. p) | p × 20,000 |

Conviene revisar cuando revisar es más barato: **p × 20,000 > (1 − p) × 1,000**. Despejando:

$$\text{umbral óptimo} = \frac{C_{FP}}{C_{FP} + C_{FN}} = \frac{1{,}000}{1{,}000 + 20{,}000} = \frac{1}{21} \approx 0.048$$

¡Hay que revisar cualquier bomba con **más de ~5 %** de probabilidad de fallar! Muy lejos del 0.5 por defecto.

> 💡 Esto supone que las probabilidades del modelo están bien **calibradas** (que "10 %" signifique de verdad 1 de cada 10). La regresión logística suele estarlo razonablemente; otros modelos no tanto. Por eso, además de la fórmula, conviene **comprobarlo con datos** (Sección 4).

### ✋ Ejercicio 3 — Calcula el umbral
Usa la fórmula C_FP / (C_FP + C_FN) para cada caso:
1. Revisión innecesaria $1,000, falla no detectada $9,000.
2. Ambos errores cuestan lo mismo.
3. Un FP cuesta **más** que un FN: FP $4,000, FN $1,000.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. 1,000 / 10,000 = **0.10**.
2. C / (C + C) = **0.5**. ¡Ese es el único caso en que el umbral por defecto es el correcto! Por eso 0.5 es el valor por defecto: supone que los errores cuestan igual.
3. 4,000 / 5,000 = **0.8**. Solo actúas si estás muy seguro: el umbral **sube** cuando el falso positivo es el error caro (piensa en el filtro de spam).
</details>

---
## 4. 📖 La curva costo vs. umbral

La fórmula da una buena primera respuesta. Ahora comprobémosla con los datos: probamos muchos umbrales y calculamos el costo de cada uno, siempre con las probabilidades de **validación cruzada** (no con el conjunto de prueba).

### 🔮 Ejercicio 4 — ¿Dónde está el fondo del valle?
1. ¿Cerca de qué umbral crees que estará el costo mínimo?
2. ¿Qué crees que pasa con el costo si el umbral es **demasiado** bajo (p. ej. 0.005)?

✍️ *Mis predicciones:*

In [ ]:
def curva_costo(prob, c_fp=C_FP, c_fn=C_FN):
    umbrales = np.linspace(0.005, 0.6, 200)
    costos = [costo(y_tr, (prob >= u).astype(int), c_fp, c_fn) for u in umbrales]
    teorico = c_fp / (c_fp + c_fn)
    plt.plot(umbrales, costos)
    plt.axvline(0.5, color="gray", ls=":", label="umbral por defecto (0.5)")
    plt.axvline(teorico, color="red", ls="--", label=f"umbral teórico ({teorico:.3f})")
    plt.xlabel("Umbral"); plt.ylabel("Costo promedio por bomba ($)"); plt.legend(); plt.grid(alpha=.3)
    plt.show()
    mejor = umbrales[np.argmin(costos)]
    print(f"Umbral con menor costo en los datos: {mejor:.3f} → ${min(costos):,.0f} por bomba")
    print(f"Umbral teórico                     : {teorico:.3f} → ${costo(y_tr, (prob >= teorico).astype(int), c_fp, c_fn):,.0f} por bomba")

curva_costo(prob)

pred = (prob >= 1 / 21).astype(int)
print(f"\nCon el umbral teórico: recall = {recall_score(y_tr, pred):.0%}, "
      f"precisión = {precision_score(y_tr, pred):.0%}, se revisa el {pred.mean():.0%} de las bombas")

<details><summary><b>👀 Ver respuesta</b></summary>

1. El mínimo de los datos está en **≈0.044** ($584 por bomba), muy cerca del teórico **0.048** ($607). La fórmula acertó casi de lleno.
2. Con un umbral demasiado bajo revisas casi todas las bombas y el costo sube hacia los $912 de "revisar todas". Con uno demasiado alto, hacia los $1,767 de "nunca revisar". El óptimo está en el fondo de un **valle**.

Con el umbral teórico el modelo detecta el **86 %** de las fallas, con una precisión de solo **17 %** (de cada 6 revisiones, 5 son innecesarias) y revisa el **44 %** de las bombas. Parece "malo" en precisión… pero es lo que **más dinero ahorra**: pasamos de $1,315 (umbral 0.5) a ~$600 por bomba.
</details>

### ✏️ Ejercicio 5 — Cambian los costos
La planta instaló bombas de repuesto y ahora una falla no detectada cuesta solo **$5,000**. Antes de ejecutar: ¿el umbral óptimo sube o baja? ¿A cuánto?

✍️ *Mi apuesta:*

In [ ]:
curva_costo(prob, c_fp=1_000, c_fn=5_000)    # ✏️ prueba otros costos

<details><summary><b>👀 Ver respuesta</b></summary>

**Sube**, a 1,000 / 6,000 ≈ **0.167**. Como la falla ahora es menos grave, ya no vale la pena revisar bombas con solo 5 % de riesgo. El mínimo de la curva se desplaza a la derecha (los datos lo ponen en **≈0.18**).

👉 **El umbral no es una propiedad del modelo: es una decisión de negocio.** El mismo modelo sirve para costos distintos; solo cambias el umbral.
</details>

---
## 5. 📖 La matriz de beneficios: cuando acertar da dinero

A veces es más natural hablar de **ganancias** que de costos. Caso: una tienda en línea decide a quién enviarle un cupón.
- Enviar un cupón cuesta **$2**.
- Si el cliente compra gracias al cupón, la tienda gana **$30** de margen.

|  | **No enviar** | **Enviar** |
|---|---|---|
| **No iba a comprar** | $0 | −$2 |
| **Sí compra** | $0 | +$30 − $2 = **+$28** |

Mismo razonamiento: enviar el cupón tiene beneficio esperado **p × 30 − 2**. Conviene si **p > 2/30 ≈ 0.067**.

### ✋ Ejercicio 6 — ¿A quién le mando el cupón?
El modelo te da estas probabilidades de compra. ¿A quiénes les enviarías el cupón y cuánto esperas ganar en total?

In [ ]:
clientes = pd.DataFrame({"prob_compra": [0.50, 0.12, 0.07, 0.05, 0.01]},
                        index=["Ana", "Beto", "Carla", "Dani", "Eva"])
COSTO_CUPON, MARGEN = 2, 30          # ✏️

clientes["ganancia_esperada"] = clientes.prob_compra * MARGEN - COSTO_CUPON
clientes["enviar?"] = np.where(clientes.ganancia_esperada > 0, "✅ sí", "❌ no")
print(clientes)
print(f"\nGanancia esperada total: ${clientes.ganancia_esperada.clip(lower=0).sum():.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

A **Ana, Beto y Carla** (probabilidad > 0.067). Ganancia esperada: 13 + 1.6 + 0.1 = **$14.70**. A Dani (5 %) no: en promedio perderías $0.50 por cupón.

Fíjate: con el umbral 0.5 solo le mandarías el cupón a Ana y dejarías de ganar $1.70. Cuando el error FP es barato, conviene un umbral **bajo**.
</details>

---
## 6. 📖 ¿Y si no sé cuánto cuesta cada error? F-beta

A menudo no tienes cifras en dinero, pero sí sabes **cuál error es peor**. Para eso existe **F-beta**, una generalización del F1 (Lección 4.2):

| `beta` | Qué privilegia | Úsalo cuando… |
|---|---|---|
| **beta < 1** (p. ej. 0.5) | la **precisión** | las falsas alarmas son lo caro (spam, recomendar algo mal) |
| **beta = 1** | ambos por igual (= F1) | no hay razón para preferir ninguno |
| **beta > 1** (p. ej. 2) | el **recall** | perder un caso es lo caro (fallas, enfermedades) |

Intuitivamente, **beta dice cuántas veces más te importa el recall que la precisión**.

### 🔮 Ejercicio 7 — Beta mueve el umbral
Buscaremos el umbral que maximiza F-beta para beta = 0.5, 1 y 2. ¿Cuál dará el umbral **más bajo**?

✍️ *Mi apuesta:*

In [ ]:
umbrales = np.linspace(0.005, 0.6, 200)
for beta in [0.5, 1, 2]:      # ✏️ prueba beta = 4
    puntajes = [fbeta_score(y_tr, (prob >= u).astype(int), beta=beta) for u in umbrales]
    mejor = umbrales[np.argmax(puntajes)]
    pred = (prob >= mejor).astype(int)
    print(f"beta = {beta:<4} → mejor umbral = {mejor:.2f}   recall = {recall_score(y_tr, pred):.0%}   "
          f"costo = ${costo(y_tr, pred):,.0f}")

<details><summary><b>👀 Ver respuesta</b></summary>

**beta = 2** da el umbral más bajo (**≈0.10**): al privilegiar el recall, el modelo "levanta la mano" más seguido. beta = 1 lo pone en ≈0.25 y beta = 0.5 en ≈0.52.

Pero mira el costo: ni siquiera beta = 2 llega a los ~$584 del umbral elegido por costo. F-beta es una **aproximación** útil cuando no tienes precios; si los tienes, **optimiza el costo directamente**.
</details>

---
## 7. 📖 Ajustar el umbral sin hacer trampa: `TunedThresholdClassifierCV`

Elegir el umbral **es** ajustar un hiperparámetro. Si lo eliges mirando el conjunto de prueba, tu resultado final deja de ser honesto (Lección 4.6). scikit-learn trae dos herramientas:

- **`FixedThresholdClassifier(modelo, threshold=...)`**: envuelve un modelo y usa el umbral que **tú** le digas (p. ej. el teórico 1/21).
- **`TunedThresholdClassifierCV(modelo, scoring=...)`**: busca **solo con validación cruzada** el umbral que optimiza tu métrica (aquí, el costo).

Ambos se comportan como un modelo normal: `.fit()` y `.predict()`. Al final evaluamos **una sola vez** en prueba.

### 🔮 Ejercicio 8 — El examen final
¿Qué esperas en el conjunto de prueba? Ordena del más barato al más caro: umbral 0.5, umbral teórico (1/21), umbral ajustado por validación cruzada, "revisar todas".

✍️ *Mi orden:*

In [ ]:
puntaje_costo = make_scorer(costo, greater_is_better=False)   # menos costo = mejor

candidatos = {
    "Logística (umbral 0.5)":  modelo,
    "Umbral teórico (1/21)":   FixedThresholdClassifier(modelo, threshold=1 / 21),
    "Umbral ajustado por CV":  TunedThresholdClassifierCV(modelo, scoring=puntaje_costo, cv=5),
}
for nombre, m in candidatos.items():
    m.fit(X_tr, y_tr)
    print(f"{nombre:<24} costo en prueba = ${costo(y_te, m.predict(X_te)):,.0f}")
print(f"{'Revisar todas':<24} costo en prueba = ${costo(y_te, np.ones_like(y_te)):,.0f}")
print(f"{'Nunca revisar':<24} costo en prueba = ${costo(y_te, np.zeros_like(y_te)):,.0f}")
print(f"\nUmbral que eligió la validación cruzada: {candidatos['Umbral ajustado por CV'].best_threshold_:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

En prueba: **ajustado por CV ($724) < teórico ($767) < revisar todas ($912) < umbral 0.5 ($1,368) < nunca revisar ($1,760)**.

- El umbral elegido por validación cruzada (≈0.040) quedó muy cerca del teórico (0.048), y los dos le ganan con holgura a 0.5.
- Con el umbral por defecto, el modelo **pierde contra la línea base** "revisar todas". Sin ajustar el umbral, habrías concluido que el modelo no sirve.
- Los costos en prueba son algo más altos que en validación cruzada (~$600): con 1,000 bombas, unas pocas fallas de más mueven el promedio. Lo importante es que el **orden** se mantiene.
</details>

---
## 8. 📖 ¿Qué métrica uso? La tabla de decisión

### Regresión (Lección 4.1)
| Situación | Métrica | Por qué |
|---|---|---|
| Todos los errores pesan según su tamaño, hay atípicos | **MAE** | Fácil de explicar ("me equivoco 5 min en promedio"); no lo dominan los atípicos |
| Los errores grandes son desproporcionadamente graves | **RMSE** | Castiga más los errores grandes |
| Los valores tienen escalas muy distintas (vender 10 o 10,000 unidades) | **MAPE** (error porcentual) | Compara en %; ¡cuidado cuando el valor real es cercano a 0! |
| Quedarse corto cuesta distinto que pasarse (inventario, personal) | **Pérdida cuantil** (*pinball*) | Apunta a un percentil, no al promedio (`mean_pinball_loss`) |
| Comparar con "predecir el promedio" | **R²** | Ya es una mejora relativa (Lección 4.7) |

### Clasificación (Lecciones 4.2 y 4.3)
| Situación | Métrica |
|---|---|
| Clases balanceadas y errores con costo parecido | **Exactitud** (siempre junto a la línea base) |
| Perder un positivo es grave (enfermedad, falla) | **Recall** (con un mínimo de precisión), **F2** |
| Las falsas alarmas son caras o molestas | **Precisión**, **F0.5** |
| Quieres equilibrio y no conoces los costos | **F1** |
| Comparar modelos **antes** de elegir umbral | **AUC-ROC**; con mucho desbalance, **AP** (área bajo la curva PR) |
| **Conoces los costos** | **El costo (o beneficio) esperado**, y ajusta el umbral con él |

> 🧠 Regla práctica: usa AUC/AP para **comparar modelos**, y el costo (o la métrica de negocio) para **elegir el umbral** y **decidir**.

### ✋ Ejercicio 9 — Empareja
¿Qué métrica principal usarías?
1. Predecir cuántas enfermeras necesita un hospital por turno; quedarse corto es mucho peor que sobrar.
2. Comparar 5 modelos de detección de una enfermedad rara (0.5 % de casos) antes de hablar con los médicos sobre el umbral.
3. Un detector de comentarios ofensivos que **oculta** automáticamente el comentario; ocultar uno inocente enfada mucho a los usuarios.
4. Predecir el precio de casas para una inmobiliaria que quiere un número fácil de explicar a sus clientes.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Pérdida cuantil** (p. ej. el percentil 90): pides "suficientes enfermeras en 9 de cada 10 turnos", no el promedio. Un modelo de regresión cuantil o `HistGradientBoostingRegressor(loss="quantile", quantile=0.9)` apunta a eso.
2. **AP** (área bajo la curva precisión-recall): con 0.5 % de positivos, la AUC-ROC puede verse optimista (Lección 4.3).
3. **Precisión** o **F0.5**: el falso positivo (ocultar algo inocente) es el error caro.
4. **MAE**: "en promedio nos equivocamos por $X" es fácil de entender. (RMSE si les preocupan especialmente los errores enormes.)
</details>

---
## 9. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Eliges el modelo con mejor exactitud y el negocio pierde dinero | La métrica no refleja el costo de los errores | Construye la matriz de costos y mide el costo esperado |
| Usas el umbral 0.5 "porque viene así" | Supone que los dos errores cuestan igual | Umbral = C_FP / (C_FP + C_FN), y compruébalo con la curva de costo |
| Ajustaste el umbral mirando el conjunto de prueba | Fuga de información: el resultado final es optimista | `TunedThresholdClassifierCV` o probabilidades de `cross_val_predict` |
| El umbral teórico funciona mal en la práctica | Probabilidades mal **calibradas** | Ajusta el umbral con datos (CV); calibra el modelo (`CalibratedClassifierCV`) |
| Nadie sabe cuánto cuesta cada error | Falta conversación con el negocio | Pide al menos un orden de magnitud ("¿10 veces peor? ¿100?") y usa F-beta mientras tanto |
| El modelo optimizado por costo no le gana a "revisar todas" | El modelo no aporta información suficiente | Compara siempre con las líneas base de costo (Lección 4.7) |
| Reportas AUC al gerente | AUC no dice cuánto se gana ni dónde poner el umbral | Al negocio, reporta el costo o beneficio esperado |

## 10. 🧠 Resumen en 5 puntos
1. La métrica correcta depende de **cuánto cuesta cada tipo de error**: no existe una métrica universal.
2. La **matriz de costos (o beneficios)** convierte la matriz de confusión en dinero y te da el **costo esperado**.
3. El **umbral óptimo** es aproximadamente **C_FP / (C_FP + C_FN)**: el 0.5 por defecto solo es correcto si ambos errores cuestan lo mismo.
4. Sin costos en dinero, **F-beta** expresa cuál error importa más (beta > 1 → recall; beta < 1 → precisión).
5. El umbral es un hiperparámetro: ajústalo con **validación cruzada** (`TunedThresholdClassifierCV`) y usa la prueba **una sola vez**.

---
## 11. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈35 min).

### 11.1 Repaso rápido (opción múltiple)

**1.** Un FP cuesta $10 y un FN cuesta $90. ¿Cuál es el umbral óptimo teórico?
- a) 0.9 · b) 0.5 · c) 0.1 · d) 0.01

**2.** Subes el umbral de 0.5 a 0.8. ¿Qué pasa normalmente?
- a) Sube el recall · b) Sube la precisión y baja el recall · c) No cambia nada · d) Sube la exactitud siempre

**3.** ¿En cuál caso usarías **F2** en lugar de F1?
- a) Filtro de spam · b) Detección de fugas de gas · c) Recomendador de películas · d) Clases balanceadas con costos iguales

**4.** Ajustas el umbral probando 100 valores en el conjunto de **prueba** y reportas el mejor costo. ¿Qué problema hay?
- a) Ninguno · b) El costo reportado será optimista · c) El modelo se vuelve más lento · d) El umbral siempre será 0.5

**5.** Tu modelo de mantenimiento con umbral 0.5 cuesta más que "revisar todas". ¿Qué deberías probar **primero**?
- a) Tirar el modelo · b) Un modelo más complejo · c) Bajar el umbral según los costos · d) Medir la exactitud

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** 10 / (10 + 90) = 0.1.
2. **b)** El modelo exige más seguridad para decir "positivo": acierta más cuando lo dice (precisión) pero deja escapar más casos (recall).
3. **b)** No detectar una fuga de gas es el error grave: hay que privilegiar el recall.
4. **b)** Elegiste el umbral que mejor le fue **a esos datos**; con datos nuevos rendirá algo peor. Es ajustar con el examen (Lección 4.6).
5. **c)** Es lo que viste en la Sección 7: el mismo modelo pasó de perder contra "revisar todas" a ganarle, solo cambiando el umbral.
</details>

### 11.2 Preguntas para pensar (sin ejecutar)

**A. Tamizaje médico 🩺** Un hospital usa un modelo para decidir quién se hace una mamografía adicional. Un FN (no detectar un cáncer) tiene un costo que **no se mide solo en dinero**. ¿Cómo elegirías el umbral? ¿Quién debería participar en esa decisión?

**B. Sin simplificar.** En la Sección 2 dijimos que una revisión útil (VP) costaba $0. En realidad cuesta $1,000 (la revisión) y evita una falla de $20,000. Si pones ese costo en la casilla VP, ¿cambia el umbral óptimo? (Pista: compara el costo de revisar contra el de no revisar.)

**C. Regresión asimétrica.** Una panadería predice cuántos panes hornear. Si sobran, pierde $1 por pan; si faltan, pierde $4 por cliente que se va sin pan. ¿Debería hornear el **promedio** de la demanda predicha? ¿Más o menos?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** No hay una cifra única, y eso está bien. Un enfoque razonable: fijar un **recall mínimo** (p. ej. "detectar al menos el 95 % de los cánceres") y, entre los umbrales que lo cumplen, elegir el de mayor precisión para no saturar al hospital. Esa decisión es **ética y clínica**, no solo técnica: deben participar médicos, pacientes y administradores. Tu trabajo es mostrarles con claridad qué se gana y qué se pierde con cada umbral.

**B.** Revisar cuesta siempre $1,000 (falle o no). No revisar cuesta p × 20,000. Conviene revisar si p × 20,000 > 1,000 → **p > 0.05**. Casi igual que 1/21 ≈ 0.048. En general la fórmula exacta usa las **diferencias** de costo entre equivocarse y acertar en cada fila: (C_FP − C_VN) / [(C_FP − C_VN) + (C_FN − C_VP)] = 1,000 / (1,000 + 19,000) = 0.05.

**C. Más** que el promedio: quedarse corto cuesta 4 veces más que pasarse. Lo óptimo es hornear el **percentil** 4 / (4 + 1) = **80 %** de la demanda (vender suficiente en 8 de cada 10 días). Es la misma lógica del umbral, en versión regresión: la **pérdida cuantil** (*pinball*) con `quantile=0.8`.
</details>

### 11.3 🧩 Práctica integradora: control de calidad en una fábrica de botellas

Una fábrica produce botellas de vidrio. Una cámara mide 5 características de cada botella y un modelo debe decidir cuáles **descartar** por defectuosas.

- Descartar una botella buena (FP) cuesta **$12** (material y energía).
- Enviar una botella defectuosa al cliente (FN) cuesta **$300** (devolución, multa del contrato y reputación).

Lo harás en 6 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos.**

In [ ]:
Xb, yb = make_classification(n_samples=3000, n_features=5, n_informative=3, n_redundant=1,
                             weights=[0.97], class_sep=1.0, flip_y=0.01, random_state=21)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.3, stratify=yb, random_state=0)
print(f"Botellas defectuosas: {yb.mean():.1%}  ({yb_tr.sum()} en entrenamiento, {yb_te.sum()} en prueba)")

C_FP_B, C_FN_B = 12, 300

def costo_botellas(y_real, y_pred):
    return costo(y_real, y_pred, c_fp=C_FP_B, c_fn=C_FN_B)

**Paso 2 🔮 — Las líneas base.** Calcula primero en papel: con 3.5 % de defectuosas, ¿cuánto cuesta por botella **no descartar nunca**? ¿Y **descartarlas todas**? Luego ejecuta.

✍️ *Mis cálculos:*

In [ ]:
print(f"Nunca descartar : ${costo_botellas(yb_tr, np.zeros_like(yb_tr)):.2f} por botella")
print(f"Descartar todas : ${costo_botellas(yb_tr, np.ones_like(yb_tr)):.2f} por botella")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Nunca descartar:** ≈ 0.035 × 300 ≈ **$10.57** por botella.
- **Descartar todas:** ≈ 0.965 × 12 ≈ **$11.58** por botella.

Las dos son carísimas (¡una botella cuesta menos que eso!). Cualquier modelo útil debe costar **mucho menos de $10.57**.
</details>

**Paso 3 ▶️ — El modelo con umbral por defecto.** Probabilidades por validación cruzada y umbral 0.5.

In [ ]:
modelo_b = make_pipeline(StandardScaler(), LogisticRegression())
prob_b = cross_val_predict(modelo_b, Xb_tr, yb_tr, cv=5, method="predict_proba")[:, 1]

pred = (prob_b >= 0.5).astype(int)
print(f"Umbral 0.5 → exactitud = {accuracy_score(yb_tr, pred):.1%}   recall = {recall_score(yb_tr, pred):.0%}   "
      f"costo = ${costo_botellas(yb_tr, pred):.2f}")

✋ ¿Qué te dice la exactitud? ¿Y el recall?

<details><summary><b>👀 Ver respuesta</b></summary>

Exactitud **97.0 %**: suena excelente, pero "nunca descartar" ya tiene 96.5 %. El recall es solo **19 %**: 4 de cada 5 botellas defectuosas llegan al cliente. Costo: **$8.59**, apenas mejor que no hacer nada.
</details>

**Paso 4 ✋ — El umbral teórico.** Calcúlalo con la fórmula y escríbelo en la celda.

In [ ]:
UMBRAL = 12 / (12 + 300)     # ✏️ comprueba la fórmula

pred = (prob_b >= UMBRAL).astype(int)
print(f"Umbral {UMBRAL:.3f} → exactitud = {accuracy_score(yb_tr, pred):.1%}   recall = {recall_score(yb_tr, pred):.0%}   "
      f"descarta el {pred.mean():.0%}   costo = ${costo_botellas(yb_tr, pred):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Umbral = 12 / 312 ≈ **0.038**. El costo baja de $8.59 a **$4.07** por botella (la mitad), el recall sube a **80 %**… y la exactitud **baja** a 83.2 %. Descartamos el 19 % de las botellas, la mayoría buenas, pero cada defectuosa atrapada ahorra $300.

Otra vez: **la exactitud empeoró y el negocio mejoró.**
</details>

**Paso 5 ▶️ — Ajuste honesto con validación cruzada.** Dejamos que `TunedThresholdClassifierCV` busque el umbral, solo con el entrenamiento.

In [ ]:
ajustado = TunedThresholdClassifierCV(modelo_b, scoring=make_scorer(costo_botellas, greater_is_better=False),
                                      cv=5).fit(Xb_tr, yb_tr)
print(f"Umbral elegido por CV: {ajustado.best_threshold_:.3f}   costo medio en CV: ${-ajustado.best_score_:.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Elige un umbral de **≈0.065**, con un costo en validación cruzada de **≈$3.53**. Es algo más alto que el teórico (0.038): las probabilidades de la logística no están perfectamente calibradas y los datos "corrigen" la fórmula. Ambos están en la misma zona, muy lejos de 0.5.
</details>

**Paso 6 ▶️ ✋ — Examen final, una sola vez.** Evaluamos en prueba y comparamos con las líneas base. Luego escribe en dos frases tu recomendación para el gerente de planta.

In [ ]:
pred_final = ajustado.predict(Xb_te)
print(f"Modelo con umbral ajustado : ${costo_botellas(yb_te, pred_final):.2f} por botella   "
      f"(recall {recall_score(yb_te, pred_final):.0%}, descarta el {pred_final.mean():.0%})")
print(f"Modelo con umbral 0.5      : ${costo_botellas(yb_te, modelo_b.fit(Xb_tr, yb_tr).predict(Xb_te)):.2f}")
print(f"Nunca descartar            : ${costo_botellas(yb_te, np.zeros_like(yb_te)):.2f}")
print(f"Descartar todas            : ${costo_botellas(yb_te, np.ones_like(yb_te)):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

En prueba: modelo ajustado **$3.83** por botella (recall 74 %, descarta el 12 %), contra **$7.36** con umbral 0.5, **$10.33** sin descartar nunca y **$11.59** descartando todas.

Un ejemplo de recomendación:

*"Recomiendo usar el modelo descartando toda botella con más de ~6.5 % de riesgo de defecto: cuesta unos $3.8 por botella, frente a $10.3 de no revisar y $7.4 del modelo con su configuración de fábrica. A cambio, desecharemos alrededor de 1 de cada 8 botellas (la mayoría buenas), lo cual vale la pena porque una botella defectuosa en el cliente cuesta 25 veces más."*
</details>

---
## 📚 Para profundizar
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Tuning the decision threshold for class prediction* (`TunedThresholdClassifierCV`, `FixedThresholdClassifier`).
- 📘 **Documentación:** scikit-learn, ejemplo *Post-tuning the decision threshold for cost-sensitive learning*.
- 📗 **Artículo clásico:** Charles Elkan (2001), *The Foundations of Cost-Sensitive Learning* (inglés): de dónde sale la fórmula del umbral.
- 📺 **Video:** StatQuest, "ROC and AUC, Clearly Explained" (para repasar umbrales, Lección 4.3).

---
⬅️ **Anterior:** [4.7 Líneas base: ¿le gano a "adivinar el promedio"?](4.7-lineas-base.ipynb) · ➡️ **Siguiente:** [4.9 Proyecto del módulo: Auditoría de un detector de fraude 🕵️](4.9-proyecto-del-modulo.md)